In [ ]:
import os
from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# A Jupyter kernel starts in the folder holding the notebook, not the repo root, so "data/..."
# here would not mean what it means in the notebooks that sit at the root. Move to the root
# once: every path below is then relative to it, and the project becomes importable, because
# the kernel resolves imports against the working directory.
if (Path.cwd() / "prepare.ipynb").is_file():
    os.chdir(Path.cwd().parents[1])
if not (Path.cwd() / "config" / "topics").is_dir():
    raise RuntimeError(f"expected the repo root, got {Path.cwd()} - run this notebook from "
                       f"its own folder (datasets/sod) or from the repo root")

from datasets.sod.code.clean import process_respondent
from datasets.sod.code.describe import render
from panel.describe.template import write_table
from panel.analysis.aggregate import explore_dataset_values


In [ ]:
SAV_PATH = "datasets/sod/SoD-CZ/Society_of_Distrust - CZ.sav"
# SAV_PATH = "datasets/sod/SoD-EN/Society_of_Distrust - EN.sav"
df = pd.read_spss(SAV_PATH)

print("\nDataset Information:")
df.info()
print(f"\nShape of the dataset (rows, columns): {df.shape}")


## Data Exploration

In [ ]:
print("\nColumn names:")
print([c for c in df.columns])

In [ ]:
row_as_series_loc = df.loc[3245]

for col,val in row_as_series_loc.items():
    print(f"{col}: {val}")

In [ ]:
base_cols = ['GENDER', 'AGE1', 'AGE2', 'EDU', 'KRAJ', 'OKRES', 'VMB', 'INCOMEP']
employment_cols = [col for col in df.columns if 'EMPLOYMENT_' in col]
question_cols = ['Q19', 'Q20', 'Q18', 'Q17', 'Q23']
columns_to_explore = base_cols + employment_cols + question_cols

explore_dataset_values(df, columns_to_explore)

In [ ]:
cols_to_plot = base_cols

# Plotting: create one figure per column for clarity, counts left, share of total right
for col in cols_to_plot:
    value_counts = df[col].dropna().astype(str).value_counts().sort_index()
    total = value_counts.sum()

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.barplot(x=value_counts.index, y=value_counts.values, ax=ax)
    ax.set_title(f"Histogram of {col} (n={total})")
    ax.set_xlabel(col)
    ax.set_ylabel("Count")
    ax.set_xticks(range(len(value_counts)), value_counts.index, rotation=45, ha='right')
    ax.set_ylim(0, value_counts.max() * 1.2)
    if len(value_counts) <= 15:
        labels = [f"{c}\n{c / total:.1%}" for c in value_counts.values]
        ax.bar_label(ax.containers[0], labels=labels, fontsize=8, padding=2)

    share = ax.twinx()
    share.set_ylim(0, ax.get_ylim()[1] / total * 100)
    share.set_ylabel("% of total")

    plt.tight_layout()
    plt.show()


## Data preparation

In [ ]:
# Indexed by the survey's own id, so descriptions, runs and results all join on identity
# rather than on row position. `age` is the exact age, `age_range` the survey's own band.
data = pd.DataFrame([process_respondent(df.loc[i]) for i in df.index]).set_index("id")
data.to_csv("data/clean_data.csv")

print(f"{len(data)} respondents -> data/clean_data.csv")
print(f"id {data.index.min()}..{data.index.max()}, unique={data.index.is_unique}")
data.head()


In [ ]:
# Rendered in the same pass that wrote the data, which is what keeps a description table
# from lagging the rows it describes. A different wording means a new variant file, never
# an edit of one a run has already recorded.
for variant in ("self_description_v1", "self_description_v2"):
    descriptions = render(data, variant)
    print(f"{len(descriptions)} descriptions -> {write_table(descriptions)}")
    print(descriptions.iloc[0], end="\n\n")


**`data/legacy_clean_data.csv` is not written here.** It is a frozen snapshot of this table as
it stood on 2026-09-08 — pre-`decapitalize`-fix values, indexed by row position — because the
ten runs under `results/election_2021_legacy/` are keyed that way and were shown those exact
values. Analysis of those runs reads it; nothing regenerates it.